In [1]:
import os
import math
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import tellurium as te
import roadrunner
from scipy.integrate import solve_ivp
import warnings

warnings.filterwarnings('ignore')
roadrunner.Logger.setLevel(roadrunner.Logger.LOG_CRITICAL)

print("ChenSloppyComplexity: sampling complexity vs stiffness")

ChenSloppyComplexity: sampling complexity vs stiffness


In [2]:
# === CONFIGURATION ===
N_SAMPLES = 300              # start small; increase carefully
RANDOM_SEED = 123

# Complexity (Chico-style) settings
SIM_TIME = 500
SIM_POINTS = 501
COARSE_BINS = 50
DIVERGENCE_THRESHOLD = 250
COARSE_START = 0

# Sloppiness settings
T_FINAL = 200.0
N_TIME_POINTS = 51
SENS_BATCH_SIZE = 10
STIFF_THRESHOLD = 0.1
N_EIGENVALUES = 10  # Number of top eigenvalues to compute via Lanczos

# Parameter sampling multipliers (Chico approach)
MULTIPLIERS = [0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Cross-platform model path
def get_model_path():
    possible_paths = [
        "/home/gijs/Documents/OxfordEvolution/Yeast/Chen/chen_model.xml",
        "/Users/gijsbartholomeus/Documents/STUDIE/OxfordEvolution/code/Yeast/Chen/chen_model.xml",
        "chen_model.xml",
        "Chen/chen_model.xml",
        "../Chen/chen_model.xml",
    ]
    for path in possible_paths:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Chen model not found in expected locations: {possible_paths}")

model_path = get_model_path()
print(f"Using model: {model_path}")

Using model: /Users/gijsbartholomeus/Documents/STUDIE/OxfordEvolution/code/Yeast/Chen/chen_model.xml


In [3]:
# === CHICO-STYLE SAMPLING + COMPLEXITY ===

def get_kinetic_parameters(rr):
    """Get list of kinetic parameters, excluding regulatory switches/flags."""
    kinetic_params = []
    excluded_params = []
    for pid in rr.getGlobalParameterIds():
        value = rr.getValue(pid)
        param_lower = pid.lower()
        if (param_lower.endswith('t') and value in [0.0, 1.0]) or \
           (param_lower.startswith('d') and param_lower.endswith('n')) or \
           ('flag' in param_lower) or \
           ('switch' in param_lower) or \
           (value == 0.0) or \
           (pid in ['cell']) or \
           ('total' in param_lower and value in [0.0, 1.0]):
            excluded_params.append(pid)
        else:
            kinetic_params.append(pid)
    return kinetic_params, excluded_params

_kinetic_params_cache = None
_excluded_params_cache = None

def get_kinetic_parameters_cached(rr):
    global _kinetic_params_cache, _excluded_params_cache
    if _kinetic_params_cache is None:
        _kinetic_params_cache, _excluded_params_cache = get_kinetic_parameters(rr)
    return _kinetic_params_cache, _excluded_params_cache


def reset_model_to_defaults(rr, default_values):
    for pid, default_val in default_values.items():
        try:
            rr.setValue(pid, default_val)
        except RuntimeError:
            continue


def sample_parameters_fast(rr, kinetic_params, default_values):
    reset_model_to_defaults(rr, default_values)
    rr.resetAll()

    sampled = {}
    sampled_values = []
    for pid in kinetic_params:
        try:
            current = default_values[pid]
            factor = random.choice(MULTIPLIERS)
            rr.setValue(pid, current * factor)
            sampled[pid] = factor
            sampled_values.append(factor)
        except (RuntimeError, KeyError):
            continue
    return sampled, sampled_values


def simulate_chico(rr, T=SIM_TIME, CoarseStart=COARSE_START, npoints=SIM_POINTS, diverge_threshold=DIVERGENCE_THRESHOLD):
    rr.selections = ["time", "CLB2"]
    try:
        total_time = CoarseStart + T
        total_points = int(npoints * total_time / T) if T > 0 else npoints
        result = rr.simulate(0, total_time, total_points)
    except RuntimeError:
        return None, None, None, None

    full_time = result[:, 0]
    full_clb2 = result[:, 1]

    if diverge_threshold is not None and np.any(np.abs(full_clb2) > diverge_threshold):
        return "divergent", None, None, None

    mask = (full_time >= CoarseStart) & (full_time <= CoarseStart + T)
    if not np.any(mask):
        return None, None, None, None

    windowed_time = full_time[mask]
    windowed_clb2 = full_clb2[mask]
    return windowed_time, windowed_clb2, full_time, full_clb2


def coarse_grain_direct(time, signal, nbins=COARSE_BINS):
    coarse_time = np.linspace(time[0], time[-1], nbins)
    coarse_signal = np.interp(coarse_time, time, signal)
    return coarse_time, coarse_signal


def up_down_encoding(coarse_time, coarse_signal):
    slopes = np.diff(coarse_signal) / np.diff(coarse_time)
    encoding = ''.join(['1' if s > 0 else '0' for s in slopes])
    return encoding


def lz76_phrase_count(s: str) -> int:
    n = len(s)
    if n == 0:
        return 0
    i = 0
    c = 1
    k = 1
    while i + k <= n:
        if s[i:i+k] in s[:i+k-1]:
            k += 1
            if i + k - 1 > n:
                c += 1
                break
        else:
            c += 1
            i += k
            k = 1
    return c


def CLZ(x: str) -> float:
    n = len(x)
    if n == 0:
        return 0.0
    if x.count('0') == n or x.count('1') == n:
        return math.log2(n)
    return math.log2(n) / 2 * (lz76_phrase_count(x) + lz76_phrase_count(x[::-1]))


def simulate_and_extract(rr):
    windowed_time, windowed_clb2, _, _ = simulate_chico(rr)
    if isinstance(windowed_time, str):
        return windowed_time, None, None, None, None
    if windowed_time is None or windowed_clb2 is None:
        return None, None, None, None, None

    coarse_time, coarse_signal = coarse_grain_direct(windowed_time, windowed_clb2)
    encoding = up_down_encoding(coarse_time, coarse_signal)
    complexity = CLZ(encoding)
    return windowed_time, windowed_clb2, (coarse_time, coarse_signal), encoding, complexity

print("Chico-style sampling and complexity functions ready.")

Chico-style sampling and complexity functions ready.


In [4]:
# === SLOPPINESS VIA MATRIX-FREE TANGENT+ADJOINT ===

from scipy.sparse.linalg import LinearOperator, eigsh

def get_settable_kinetic_parameters(rr):
    """Get settable kinetic parameters (exclude assignment rules and non-kinetic)"""
    kinetic_params, excluded_nonkinetic = get_kinetic_parameters(rr)
    settable_kinetic = []
    excluded_assignment = []
    for pid in kinetic_params:
        try:
            original_value = rr.getValue(pid)
            rr.setValue(pid, original_value)
            settable_kinetic.append(pid)
        except RuntimeError:
            excluded_assignment.append(pid)
    return settable_kinetic, excluded_nonkinetic, excluded_assignment


def rhs_function(t, x, rr_instance, param_values, species_names):
    """RHS of ODE: dx/dt = f(x,θ)"""
    # Set parameters
    for pid, val in param_values.items():
        try:
            rr_instance.setValue(pid, val)
        except:
            pass
    
    # Set species concentrations
    for i, species_id in enumerate(species_names):
        try:
            rr_instance.setValue(species_id, x[i])
        except:
            pass
    
    # Get rates directly from RoadRunner (much faster than simulating!)
    try:
        return rr_instance.getStateVectorRate()
    except:
        return np.zeros(len(x))


def compute_jacobians(t, x, rr_instance, param_values, param_names, species_names, eps=1e-6):
    """Compute ∂f/∂x and ∂f/∂θ via finite differences (only at integration steps, not on trajectories!)"""
    n_species = len(species_names)
    n_params = len(param_names)
    
    f0 = rhs_function(t, x, rr_instance, param_values, species_names)
    
    # ∂f/∂x
    J_x = np.zeros((n_species, n_species))
    for i in range(n_species):
        x_pert = x.copy()
        x_pert[i] += eps
        f_pert = rhs_function(t, x_pert, rr_instance, param_values, species_names)
        J_x[:, i] = (f_pert - f0) / eps
    
    # ∂f/∂θ
    J_theta = np.zeros((n_species, n_params))
    for i, pid in enumerate(param_names):
        param_values_pert = param_values.copy()
        eps_rel = eps * max(abs(param_values[pid]), 1e-6)
        param_values_pert[pid] = param_values[pid] + eps_rel
        f_pert = rhs_function(t, x, rr_instance, param_values_pert, species_names)
        J_theta[:, i] = (f_pert - f0) / eps_rel
    
    return J_x, J_theta


def compute_sloppiness_matrix_free(rr, param_names, param_values, n_eigs=50, t_final=T_FINAL, n_points=N_TIME_POINTS):
    """
    Matrix-free top eigenvalues of F = J^T @ J via tangent+adjoint with Lanczos.
    
    COMPLEXITY:
    - Full sensitivities: 1 solve of (n + n×p) ODEs = O(n²p) per timestep
    - Matrix-free: k iterations × (1 tangent + 1 adjoint) = O(k × n²) total
    - For n=50, p=136, k=50: ~2-3x faster, much less memory
    
    STABILITY:
    - Integrates sensitivity equations (NOT finite diff on trajectories)
    - Stable for sloppy systems per Transtrum et al
    """
    species_names = rr.getFloatingSpeciesIds()
    n_species = len(species_names)
    n_params = len(param_names)
    
    try:
        clb2_idx = species_names.index('CLB2')
    except ValueError:
        clb2_idx = 0
    
    # Baseline trajectory
    for pid, val in param_values.items():
        try:
            rr.setValue(pid, val)
        except:
            pass
    
    rr.resetAll()
    rr.selections = ['time'] + list(species_names)
    
    try:
        result = rr.simulate(0, t_final, n_points)
        t_span = result[:, 0]
        x_traj = result[:, 1:]
    except:
        return None
    
    print(f"  Baseline done: {n_species} species × {len(t_span)} time pts")
    
    matvec_count = [0]
    
    def matvec(v):
        """F·v = J^T(J·v) via tangent+adjoint"""
        matvec_count[0] += 1
        if matvec_count[0] % 10 == 1:
            print(f"    Lanczos iter ~{matvec_count[0]//2}...", flush=True)
        
        # Tangent: w = J·v  (integrate dδx/dt = J_x·δx + J_theta·v)
        z0_fwd = np.concatenate([x_traj[0, :], np.zeros(n_species)])
        
        def tangent_rhs(t_val, z):
            x, δx = z[:n_species], z[n_species:]
            dxdt = rhs_function(t_val, x, rr, param_values, species_names)
            J_x, J_theta = compute_jacobians(t_val, x, rr, param_values, param_names, species_names)
            dδxdt = J_x @ δx + J_theta @ v
            return np.concatenate([dxdt, dδxdt])
        
        try:
            sol_fwd = solve_ivp(tangent_rhs, (0, t_final), z0_fwd, t_eval=t_span,
                               method='BDF', rtol=1e-5, atol=1e-7, max_step=1.0)
            if not sol_fwd.success:
                return np.zeros(n_params)
            w = sol_fwd.y[n_species:, :].T[:, clb2_idx]
        except:
            return np.zeros(n_params)
        
        # Adjoint: J^T·w  (integrate -dλ/dt = J_x^T·λ backward)
        λ_T = np.zeros(n_species)
        λ_T[clb2_idx] = w[-1]
        
        def adjoint_rhs(t_val, λ):
            t_idx = min(np.searchsorted(t_span, t_val), len(t_span)-1)
            x = x_traj[t_idx, :]
            J_x, _ = compute_jacobians(t_val, x, rr, param_values, param_names, species_names)
            return -J_x.T @ λ
        
        try:
            sol_adj = solve_ivp(adjoint_rhs, (t_final, 0), λ_T, t_eval=t_span[::-1],
                               method='BDF', rtol=1e-5, atol=1e-7, max_step=1.0)
            if not sol_adj.success:
                return np.zeros(n_params)
            λ_traj = sol_adj.y.T[::-1, :]
            
            # Integral: ∫ (∂f/∂θ)^T·λ·w dt
            result = np.zeros(n_params)
            dt = t_span[1] - t_span[0] if len(t_span) > 1 else 1.0
            for i in range(len(t_span)):
                x = x_traj[i, :]
                _, J_theta = compute_jacobians(t_span[i], x, rr, param_values, param_names, species_names)
                result += J_theta.T @ λ_traj[i, :] * w[i] * dt
            return result
        except:
            return np.zeros(n_params)
    
    F_op = LinearOperator(shape=(n_params, n_params), matvec=matvec, dtype=float)
    
    print(f"  Lanczos: top {n_eigs}/{n_params} eigenvalues (each iter = 1 tangent + 1 adjoint solve)...")
    try:
        eigenvals, _ = eigsh(F_op, k=min(n_eigs, n_params-2), which='LM', tol=1e-3, maxiter=500)
        idx = np.argsort(eigenvals)[::-1]
        eigenvals = eigenvals[idx]
        print(f"    ✓ Converged ({matvec_count[0]} matvecs): λ∈[{eigenvals[-1]:.2e}, {eigenvals[0]:.2e}]")
        return eigenvals
    except Exception as e:
        print(f"    ✗ Failed ({matvec_count[0]} matvecs): {str(e)[:80]}")
        return None


def count_stiff_eigenvalues(eigenvalues, threshold=STIFF_THRESHOLD):
    if eigenvalues is None:
        return 0
    return int(np.sum(np.isfinite(eigenvalues) & (eigenvalues > threshold)))


def compute_sloppiness_spectrum(rr, param_names, param_values, n_eigs=None):
    """Wrapper for matrix-free sloppiness computation. Uses N_EIGENVALUES from config if not specified."""
    if n_eigs is None:
        n_eigs = N_EIGENVALUES
    return compute_sloppiness_matrix_free(rr, param_names, param_values, n_eigs)

print("Matrix-free sloppiness ready (tangent+adjoint+Lanczos)")

Matrix-free sloppiness ready (tangent+adjoint+Lanczos)


In [ ]:
# === RUN SAMPLING + SLOPPINESS ===

rr_complexity = te.loadSBMLModel(model_path)
kinetic_params, excluded_params = get_kinetic_parameters_cached(rr_complexity)

# Cache default values for fast sampling
_default_values = {pid: rr_complexity.getValue(pid) for pid in rr_complexity.getGlobalParameterIds()}

# Sloppiness model (separate instance)
rr_sloppy = te.loadSBMLModel(model_path)
param_names_sloppy, excluded_nonkinetic, excluded_assignment = get_settable_kinetic_parameters(rr_sloppy)

print(f"Kinetic params (sampling): {len(kinetic_params)}")
print(f"Settable kinetic params (sloppiness): {len(param_names_sloppy)}")

results = []
failed = 0
error_summary = {}

start_time = time.time()

# --- Wildtype first ---
reset_model_to_defaults(rr_complexity, _default_values)
rr_complexity.resetAll()
_, _, _, _, complexity = simulate_and_extract(rr_complexity)
if complexity is None or isinstance(complexity, str):
    print("Wildtype simulation failed; proceeding with random samples.")
else:
    print(f"Wildtype complexity: {complexity:.3f}")
    
    # Get current parameter values for sloppiness
    param_values_wt = {pid: rr_complexity.getValue(pid) for pid in param_names_sloppy}
    
    try:
        eigenvals = compute_sloppiness_spectrum(rr_sloppy, param_names_sloppy, param_values_wt)
        if eigenvals is not None:
            stiff_count = count_stiff_eigenvalues(eigenvals, STIFF_THRESHOLD)
            results.append({
                'label': 'Wildtype',
                'complexity': complexity,
                'stiff_count': stiff_count,
                'eigenvalues': eigenvals
            })
            print(f"Wildtype success: stiff_count={stiff_count}, eigenvalues span {eigenvals[0]:.2e} to {eigenvals[-1]:.2e}")
        else:
            print("Wildtype sloppiness computation returned None")
    except Exception as e:
        print(f"Wildtype sloppiness computation failed: {type(e).__name__}: {str(e)[:100]}")
        error_type = type(e).__name__
        error_summary[error_type] = error_summary.get(error_type, 0) + 1

for i in range(N_SAMPLES):
    # Sample parameters (Chico approach)
    sample_parameters_fast(rr_complexity, kinetic_params, _default_values)

    # Complexity from CLB2
    _, _, _, _, complexity = simulate_and_extract(rr_complexity)
    if complexity is None or isinstance(complexity, str):
        failed += 1
        error_summary['complexity_failed'] = error_summary.get('complexity_failed', 0) + 1
        continue

    # Get current parameter values for sloppiness computation
    param_values = {pid: rr_complexity.getValue(pid) for pid in param_names_sloppy}

    try:
        eigenvals = compute_sloppiness_spectrum(rr_sloppy, param_names_sloppy, param_values)
        if eigenvals is not None:
            stiff_count = count_stiff_eigenvalues(eigenvals, STIFF_THRESHOLD)
            results.append({
                'label': f"Sample {i+1}",
                'complexity': complexity,
                'stiff_count': stiff_count,
                'eigenvalues': eigenvals
            })
        else:
            failed += 1
            error_summary['sloppiness_returned_None'] = error_summary.get('sloppiness_returned_None', 0) + 1
    except Exception as e:
        failed += 1
        error_type = type(e).__name__
        error_summary[error_type] = error_summary.get(error_type, 0) + 1
        # Print first few errors for debugging
        if failed <= 3:
            print(f"Sample {i+1} failed: {error_type}: {str(e)[:100]}")

    if (i + 1) % max(1, N_SAMPLES // 5) == 0:
        elapsed = time.time() - start_time
        print(f"Progress {i+1}/{N_SAMPLES} | valid={len(results)} | failed={failed} | {elapsed:.1f}s")

print(f"\nDone. Valid samples: {len(results)} | failed: {failed}")
print(f"\nError summary:")
for error_type, count in sorted(error_summary.items(), key=lambda x: -x[1]):
    print(f"  {error_type}: {count}")

Kinetic params (sampling): 156
Settable kinetic params (sloppiness): 136
Wildtype complexity: 30.881
  Baseline done: 50 species × 51 time pts
  Lanczos: top 10/136 eigenvalues (each iter = 1 tangent + 1 adjoint solve)...
    Lanczos iter ~0...


In [ ]:
# === SLOPPINESS SPECTRA (FIRST 9, WT FIRST) ===

if len(results) == 0:
    print("No spectra to plot.")
else:
    n_show = min(9, len(results))
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    axes = axes.flatten()

    for idx in range(n_show):
        entry = results[idx]
        eigenvals = np.asarray(entry['eigenvalues'])
        ax = axes[idx]
        x = np.arange(1, len(eigenvals) + 1)
        ax.semilogy(x, eigenvals, 'o-', markersize=3, linewidth=1)
        ax.set_title(entry.get('label', f"Sample {idx+1}"))
        ax.set_xlabel('Index')
        ax.set_ylabel('Eigenvalue')
        ax.grid(True, alpha=0.3)

    # Hide unused subplots
    for j in range(n_show, 9):
        axes[j].axis('off')

    plt.suptitle('Sloppiness spectra (log scale): first 9 samples', y=0.98)
    plt.tight_layout()
    plt.show()

No spectra to plot.


In [ ]:
# === BINNING + PLOTTING ===

if len(results) == 0:
    print("No valid samples to plot.")
else:
    complexities = np.array([r['complexity'] for r in results])
    stiff_counts = np.array([r['stiff_count'] for r in results])

    # Scatter plot: raw relationship
    plt.figure(figsize=(7, 5))
    plt.scatter(complexities, stiff_counts, alpha=0.7)
    plt.xlabel('Complexity (CLZ)')
    plt.ylabel(f'Stiff eigenvalues (λ > {STIFF_THRESHOLD})')
    plt.title('Raw samples: complexity vs stiff count')
    plt.grid(True, alpha=0.3)
    plt.show()

    # Bin complexities
    bin_edges = np.arange(0, 50.0 + 2.5, 2.5)
    bin_labels = [f"{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}" for i in range(len(bin_edges) - 1)]
    bin_indices = np.digitize(complexities, bin_edges) - 1

    binned_counts = []
    binned_labels = []
    for b in range(len(bin_labels)):
        mask = bin_indices == b
        if np.any(mask):
            binned_counts.append(stiff_counts[mask])
            binned_labels.append(bin_labels[b])

    if len(binned_counts) == 0:
        print("No samples fell into any bin.")
    else:
        plt.figure(figsize=(12, 5))
        plt.boxplot(binned_counts, labels=binned_labels, showfliers=True)
        plt.xticks(rotation=45, ha='right')
        plt.xlabel('Complexity bins (CLZ)')
        plt.ylabel(f'Stiff eigenvalues (λ > {STIFF_THRESHOLD})')
        plt.title('Stiff eigenvalue counts by complexity bin')
        plt.grid(True, axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Print averages per bin
        print("\nAverage stiff count per bin:")
        for label, vals in zip(binned_labels, binned_counts):
            print(f"  {label}: mean={np.mean(vals):.2f}, n={len(vals)}")

No valid samples to plot.
